# Malicious_TLS 전처리 파이프라인
**캡스톤디자인 I** - 특징 추출 최적화 기반 암호화된 공격 트래픽 탐지 및 분류 시스템
팀장: 김강민 | 팀원: 서은빈

---
### 노이즈 제거 목록
| 단계 | 처리 내용 |
|---|---|
| 1 | payload_flow_duration = 0 제거 |
| 2 | 음수값 행 제거 (IP_/payload_ 계열 버그) |
| 3 | 극단값 행 제거 (99.9th percentile 초과) |
| 4 | 중복 행 제거 |
| 5 | tls_* 16진수 문자열 → 정수 변환 |
| 6 | tls_version 원핫 인코딩 |

### Google Drive 폴더 구조
```
capstone2026/
├── Malicious_TLS/
│   └── malicious_TLS.csv
└── output/
    └── malicious_tls_cleaned.csv
```

## 0. Google Drive 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR   = '/content/drive/MyDrive/capstone2026'
DATA_PATH  = os.path.join(BASE_DIR, 'Malicious_TLS/malicious_TLS.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'DATA_PATH : {DATA_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
DATA_PATH : /content/drive/MyDrive/capstone2026/Malicious_TLS/malicious_TLS.csv
OUTPUT_DIR: /content/drive/MyDrive/capstone2026/output


## 1. 라이브러리 임포트

In [2]:
import pandas as pd
import numpy as np

print('임포트 완료')

임포트 완료


## 2. 설정값

In [10]:
# 음수 불가 컬럼 (음수 = 버그)
NEGATIVE_INVALID_COLS = [
    'IP_flow_duration',
    'IP_total_packets_sent_rate',
    'IP_fwd_packets_sent_rate',
    'IP_backward_packets_sent_rate',
    'IP_total_bytes_sent_rate',
    'IP_fwd_bytes_sent_rate',
    'IP_backward_bytes_sent_rate',
    'IP_fwd_arrtime_diff_Min',        # ← 추가
    'IP_backward_arrtime_diff_Min',   # ← 추가
    'payload_flow_duration',
    'payload_total_packets_sent_rate',
    'payload_fwd_packets_sent_rate',
    'payload_backward_packets_sent_rate',
    'payload_total_bytes_sent_rate',
    'payload_fwd_bytes_sent_rate',
    'payload_backward_bytes_sent_rate',
    'payload_fwd_arrtime_diff_Min',       # ← 추가
    'payload_fwd_arrtime_diff_Mean',      # ← 추가
    'payload_backward_arrtime_diff_Min',  # ← 추가
    'payload_backward_arrtime_diff_Mean', # ← 추가
]

# 극단값 제거 기준 (99.9th percentile 초과)
OUTLIER_COLS = [
    'IP_flow_duration',
    'IP_total_backward_packets',
    'IP_total_packets',
    'IP_total_backward_bytes',
    'IP_total_bytes',
]

# 16진수 문자열 컬럼 → 정수 변환
HEX_COLS = [
    'tls_version',
    'tls_chosen_cipher_suit',
    'tls_cipher_suites_0',
    'tls_cipher_suites_1',
    'tls_cipher_suites_2',
    'tls_cipher_suites_3',
    'tls_cipher_suites_4',
]

print('설정 완료')

설정 완료


## 3. 데이터 로드

In [11]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print(f'[로드 완료] {len(df):,}행 | {len(df.columns)}컬럼')

[로드 완료] 92,034행 | 118컬럼


## 4. 노이즈 제거

In [12]:
def remove_noise(df):
    original_len = len(df)
    log = []

    # 노이즈 1: payload_flow_duration = 0 제거
    before = len(df)
    df = df[df['payload_flow_duration'] != 0]
    log.append(f'payload_flow_duration=0 제거: {before - len(df):,}행')

    # 노이즈 2: 음수값 행 제거
    before = len(df)
    neg_cols = [c for c in NEGATIVE_INVALID_COLS if c in df.columns]
    neg_mask = (df[neg_cols] < 0).any(axis=1)
    df = df[~neg_mask]
    log.append(f'음수값 행 제거: {before - len(df):,}행')

    # 노이즈 3: 극단값 행 제거 (99.9th percentile 초과)
    before = len(df)
    outlier_cols = [c for c in OUTLIER_COLS if c in df.columns]
    thresholds = df[outlier_cols].quantile(0.999)
    outlier_mask = (df[outlier_cols] > thresholds).any(axis=1)
    df = df[~outlier_mask]
    log.append(f'극단값 행 제거 (99.9th 초과): {before - len(df):,}행')

    # 노이즈 4: 중복 행 제거
    before = len(df)
    df = df.drop_duplicates()
    log.append(f'중복 행 제거: {before - len(df):,}행')

    print('[노이즈 제거]')
    for l in log:
        print(f'  ✓ {l}')
    print(f'  → {original_len:,}행 → {len(df):,}행 (제거: {original_len - len(df):,}행)')
    return df.reset_index(drop=True)

df = remove_noise(df)

[노이즈 제거]
  ✓ payload_flow_duration=0 제거: 4행
  ✓ 음수값 행 제거: 9행
  ✓ 극단값 행 제거 (99.9th 초과): 246행
  ✓ 중복 행 제거: 850행
  → 92,034행 → 90,925행 (제거: 1,109행)


## 5. TLS 특징 인코딩

In [13]:
def encode_tls_features(df):
    print('[TLS 특징 인코딩]')

    # 16진수 문자열 → 정수 변환
    hex_cols = [c for c in HEX_COLS if c in df.columns]
    for col in hex_cols:
        df[col] = df[col].apply(
            lambda x: int(str(x), 16) if pd.notna(x) and str(x).startswith('0x') else 0
        ).astype('int32')
        print(f'  ✓ {col}: 16진수 → 정수 변환')

    return df

df = encode_tls_features(df)

[TLS 특징 인코딩]
  ✓ tls_version: 16진수 → 정수 변환
  ✓ tls_chosen_cipher_suit: 16진수 → 정수 변환
  ✓ tls_cipher_suites_0: 16진수 → 정수 변환
  ✓ tls_cipher_suites_1: 16진수 → 정수 변환
  ✓ tls_cipher_suites_2: 16진수 → 정수 변환
  ✓ tls_cipher_suites_3: 16진수 → 정수 변환
  ✓ tls_cipher_suites_4: 16진수 → 정수 변환


## 6. 최종 검증

In [14]:
print('[최종 검증]')

num_cols   = df.select_dtypes(include=[np.number]).columns
nan_remain = int(df.isnull().sum().sum())
inf_remain = int(np.isinf(df[num_cols]).sum().sum())
dup_remain = int(df.duplicated().sum())

# 음수 잔여 (0~29 컬럼 제외 - 차분값이라 음수 정상)
check_cols = [c for c in num_cols if c.startswith('IP_') or c.startswith('payload_')]
neg_remain = int((df[check_cols] < 0).sum().sum())

print(f'  NaN 잔여     : {nan_remain}개  {"✅" if nan_remain == 0 else "❌"}')
print(f'  Inf 잔여     : {inf_remain}개  {"✅" if inf_remain == 0 else "❌"}')
print(f'  음수값 잔여  : {neg_remain}개  {"✅" if neg_remain == 0 else "❌"}')
print(f'  중복 행 잔여 : {dup_remain}개  {"✅" if dup_remain == 0 else "❌"}')

print(f'\n[최종 데이터셋]')
print(f'  행 수  : {len(df):,}')
print(f'  특징 수: {len(df.columns) - 1}개 (Label 제외)')

print(f'\n[최종 클래스 분포]')
vc = df['Label'].value_counts()
total = len(df)
for k, v in vc.items():
    print(f'  {k}: {v:,} ({v/total*100:.2f}%)')

[최종 검증]
  NaN 잔여     : 0개  ✅
  Inf 잔여     : 0개  ✅
  음수값 잔여  : 0개  ✅
  중복 행 잔여 : 0개  ✅

[최종 데이터셋]
  행 수  : 90,925
  특징 수: 117개 (Label 제외)

[최종 클래스 분포]
  benign: 33,213 (36.53%)
  burpsuite_burpsuite_TLS_scan: 4,798 (5.28%)
  Vawtrak.C_None_TLS_CC: 3,283 (3.61%)
  Tiggre_None_TLS_CC: 3,266 (3.59%)
  Shifu.A_None_TLS_CC: 3,255 (3.58%)
  Tor_None_TLS_CC: 3,182 (3.50%)
  Caphaw.A_None_TLS_CC: 3,124 (3.44%)
  Caphaw.AH_None_TLS_CC: 3,062 (3.37%)
  Drixed_None_TLS_CC: 2,996 (3.30%)
  BuerLoader_None_TLS_CC: 2,987 (3.29%)
  TrickBotCC_None_TLS_CC: 2,980 (3.28%)
  CobaltStrike_None_TLS_CC: 2,952 (3.25%)
  DridexCC_None_TLS_CC: 2,877 (3.16%)
  PandaZeuSCC_None_TLS_CC: 2,782 (3.06%)
  Totbrick_None_TLS_CC: 2,708 (2.98%)
  Dynamer!ac_None_TLS_CC: 2,266 (2.49%)
  golistmero_golistmero_TLS_scan: 1,999 (2.20%)
  nessus_nessus_TLS_scan: 1,998 (2.20%)
  arachni_arachni_TLS_scan: 1,810 (1.99%)
  Banker_None_TLS_CC: 1,780 (1.96%)
  Panda.BZA!tr_None_TLS_CC: 1,313 (1.44%)
  Qakbot_None_TLS_CC: 1,196 (1.32

## 7. 저장

In [16]:
out_path = os.path.join(OUTPUT_DIR, 'malicious_tls_cleaned.csv')
df.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'[저장 완료] {out_path}')
print(f'  최종: {len(df):,}행 | 특징: {len(df.columns)-1}개')

[저장 완료] /content/drive/MyDrive/capstone2026/output/malicious_tls_cleaned.csv
  최종: 90,925행 | 특징: 117개
